# Computing an Exact Interaction

This notebook showcases how for a simple 3 player game a single interaction is computed.

In [1]:
from dataclasses import dataclass
from typing import Any, ClassVar

import jax
import jax.numpy as jnp
from jaxtyping import Float, Shaped, UInt8
from ndlike.protected_axis.jax import JaxAxisProtected

from shapiq.game import Game, PackedFinitePlayerGame
from shapiq.players import FinitePlayers
from shapiq.typing import typechecked

In [2]:
@typechecked
@dataclass
class TableGame(
    PackedFinitePlayerGame[Any, Float[jax.Array, "..."]], JaxAxisProtected[jax.Array]
):
    values: Float[jax.Array, "*batch 2**{_players.ids.shape[-1]}"]
    _players: Shaped[FinitePlayers, "#*batch"]
    protected_axes: ClassVar[dict[str, int]] = {"values": 1}

    @classmethod
    def _pytree_child_names(cls) -> tuple[str, ...]:
        # Players are dynamic, but not an equally batched protected field.
        return ("values", "_players")

    @property
    @typechecked
    def players(self: Shaped[Game, "*batch"]) -> Shaped[FinitePlayers, "#*batch"]:
        return self._players

    @typechecked
    def _eval_packed(
        self: Shaped[Game, "#*batch"],
        coalition: UInt8[jax.Array, "#*batch ({self.players.n_players}+7)//8"],
    ) -> Float[jax.Array, "*batch"]:
        n_players = self.players.n_players
        index_dtype = jnp.uint64 if jax.config.x64_enabled else jnp.uint32
        if n_players > jnp.iinfo(index_dtype).bits:
            msg = "Table indices exceed the enabled integer width."
            raise ValueError(msg)
        shifts = 8 * jnp.arange(coalition.shape[-1], dtype=index_dtype)
        index = jnp.sum(coalition.astype(index_dtype) << shifts, axis=-1, dtype=index_dtype)
        index &= jnp.array((1 << n_players) - 1, dtype=index_dtype)
        shape = jnp.broadcast_shapes(self.shape, index.shape)
        table = jnp.broadcast_to(self.values, (*shape, self.values.shape[-1]))
        index = jnp.broadcast_to(index, shape)
        return jnp.take_along_axis(table, index[..., None], axis=-1)[..., 0]

In [3]:
coalitions = jnp.array(
    [
        [0, 0, 0],  # empty
        [1, 0, 0],  # {0}
        [0, 1, 0],  # {1}
        [1, 1, 0],  # {0, 1}
        [0, 0, 1],  # {2}
        [1, 0, 1],  # {0, 2}
        [0, 1, 1],  # {1, 2}
        [1, 1, 1],  # {0, 1, 2}
    ],
    dtype=bool,
)
g = TableGame(jnp.array([[0.0, 1.0, 2.0, 4.0, 3.0, 5.0, 6.0, 10.0]]), FinitePlayers(3))
print("game batch:", g.shape, "shared players:", g.players.shape)
for coalition, value in zip(coalitions.tolist(), g(coalitions).tolist(), strict=True):
    print(coalition, "->", value)

game batch: (1,) shared players: ()
[False, False, False] -> 0.0
[True, False, False] -> 1.0
[False, True, False] -> 2.0
[True, True, False] -> 4.0
[False, False, True] -> 3.0
[True, False, True] -> 5.0
[False, True, True] -> 6.0
[True, True, True] -> 10.0


Now we take a weighting scheme and put it into the discrete derivative computation

In [4]:
coalitions.shape

(8, 3)

In [5]:
# Shapley Implementation
from shapiq.game.discretederivative import DiscreteDerivativeInteractionIndex
from shapiq.game.weights import DDShapleyWeight

shapleyweight = DDShapleyWeight()
ddcomputer = DiscreteDerivativeInteractionIndex(weights=shapleyweight,
                                                game=g,
                                                reference_coalition=jnp.array([[1,0,0],[0,1,0],[0,0,1]], dtype=bool),
                                                order=1
                                                )
# We need to broadcast the coalition such that its output can be broadcasted with the interaction batch
ddshapley = ddcomputer(coalition=coalitions[:,None,:])
shapley_values = jnp.sum(ddshapley, axis=0)
shapley_values, shapley_values.sum(), ddshapley.shape

(Array([2.333334 , 3.3333342, 4.3333344], dtype=float32),
 Array(10.000002, dtype=float32),
 (8, 3))

In [6]:
# Banzhaf Implementation
from shapiq.game.discretederivative import DiscreteDerivativeInteractionIndex
from shapiq.game.weights import DDBanzhafWeight

shapleyweight = DDBanzhafWeight()
ddcomputer = DiscreteDerivativeInteractionIndex(weights=shapleyweight,
                                                game=g,
                                                reference_coalition=jnp.array([[1,0,0],[0,1,0],[0,0,1]], dtype=bool),
                                                order=1)
ddbanzhaf = ddcomputer(coalition=coalitions[:,None,:])
banzhaf_values = jnp.sum(ddbanzhaf, axis=0)
banzhaf_values, banzhaf_values.sum(), ddbanzhaf.shape

(Array([2.25, 3.25, 4.25], dtype=float32), Array(9.75, dtype=float32), (8, 3))

In [7]:
# Banzhaf Implementation
from shapiq.game.discretederivative import DiscreteDerivativeInteractionIndex
from shapiq.game.weights import DDWeightedBanzhafWeight

shapleyweight = DDWeightedBanzhafWeight(weight=jnp.array([[0.5],[0.3]]))
ddcomputer = DiscreteDerivativeInteractionIndex(weights=shapleyweight,
                                                game=g,
                                                reference_coalition=jnp.array([[1,0,0],[0,1,0],[0,0,1]], dtype=bool),
                                                order=1)
weighed_banzhaf = ddcomputer(coalition=coalitions[:,None,None,:])
weighed_banzhaf_values = jnp.sum(weighed_banzhaf, axis=0)
weighed_banzhaf_values, weighed_banzhaf_values.sum(), weighed_banzhaf.shape

(Array([[2.25     , 3.25     , 4.25     ],
        [1.6899999, 2.6899998, 3.69     ]], dtype=float32),
 Array(17.82, dtype=float32),
 (8, 2, 3))